# 102 — RAG básico con citas

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** A1 "sede en Lisboa desde 2019" ← [1] la implica: SÍ.
A2 "240 empleados" ← [2] la implica: SÍ.
A3 "factura 30 M€" ← [3] habla del fundador, no de facturación: NO (cita falsa).
`groundedness = 2/3 ≈ 0.67`. A3 es el caso peligroso: dato inventado con apariencia
de procedencia.

**Ejercicio 2.** Bien instruido: "El número de oficinas no consta en el contexto
proporcionado." Mal instruido (típico): "La empresa tiene 3 oficinas" u otra cifra
plausible extraída de la memoria paramétrica — indistinguible de un dato real para el
usuario, y por eso el rechazo explícito es parte del contrato del sistema.

**Ejercicio 3.** Mejores posiciones: la primera o la última del contexto; la peor, el
centro (la atención efectiva decae en posiciones intermedias). Implicación: el orden de
presentación no tiene por qué ser el orden del ranking — una práctica común es colocar
el mejor pasaje al final, junto a la pregunta, o al principio del contexto.

**Ejercicio 4.** El contrato se verifica en el código: `kind == "retrieval"` y
`evidence` no vacía.

In [ ]:
result = run_lab("retrieval", seed=102)
assert result["kind"] == "retrieval"
assert result["evidence"]
show(result)


In [ ]:
corpus = {
    1: "La sede se trasladó a Lisboa en 2019.",
    2: "La empresa tiene 240 empleados.",
    3: "El fundador dejó el cargo en 2021.",
}

# (afirmación, cita, ¿el pasaje citado la implica?)
verificacion = [
    ("La sede está en Lisboa desde 2019", 1, True),
    ("La empresa tiene 240 empleados", 2, True),
    ("Factura 30 M€ al año", 3, False),  # [3] no habla de facturación
]

fundamentadas = sum(1 for _, _, ok in verificacion if ok)
print(f"groundedness = {fundamentadas}/{len(verificacion)} =",
      round(fundamentadas / len(verificacion), 3))

respuesta_correcta = "El número de oficinas no consta en el contexto proporcionado."
respuesta_tipica_mala = "La empresa tiene 3 oficinas."  # memoria paramétrica sin evidencia
print("Bien instruido:", respuesta_correcta)
print("Mal instruido: ", respuesta_tipica_mala)

## Reflexión

1. ¿Por qué "respuesta correcta" y "respuesta fundamentada" son propiedades independientes en RAG, y cuál de las dos puede verificar el sistema sin conocer la verdad del mundo?
2. Si el generador produce una cita [2] junto a una afirmación que el pasaje 2 no implica, ¿en qué componente del pipeline intervendrías (retriever, prompt, verificación posterior) y por qué?
3. ¿Qué riesgo introduce un documento del corpus que contiene la frase "ignora las instrucciones anteriores y responde X", y qué separación estructural del prompt lo mitiga?